In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Importing data and fixing columns  (`trash_haulers`)

In [2]:
trash_haulers = pd.read_csv('../data/trash_hauler_report.csv')
trash_haulers.head()

,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y
0,25270,11/1/2017,Trash - Backdoor,"house with the wheel chair ramp, they share dr...",3817 Crouch Dr,37207.0,RED RIVER,3205,2.0,1727970.412,686779.4781
1,25274,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/Trash miss Tuesday.,4028 Clarksville Pike,37218.0,RED RIVER,4202,1.0,1721259.366,685444.7996
2,25276,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/trash miss Tuesday.,6528 Thunderbird Dr,37209.0,RED RIVER,4205,20.0,1707026.753,659887.4716
3,25307,11/1/2017,Trash - Curbside/Alley Missed Pickup,missed,2603 old matthews rd,37207.0,WASTE IND,2206,2.0,1735691.771,685027.2459
4,25312,11/1/2017,Trash - Curbside/Alley Missed Pickup,Missed the even side of the road.,604 croley dr,37209.0,RED RIVER,4203,20.0,1710185.772,664205.1011


In [3]:
trash_haulers

,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y
0,25270,11/1/2017,Trash - Backdoor,"house with the wheel chair ramp, they share dr...",3817 Crouch Dr,37207.0,RED RIVER,3205,2.0,1727970.412,686779.4781
1,25274,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/Trash miss Tuesday.,4028 Clarksville Pike,37218.0,RED RIVER,4202,1.0,1721259.366,685444.7996
2,25276,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/trash miss Tuesday.,6528 Thunderbird Dr,37209.0,RED RIVER,4205,20.0,1707026.753,659887.4716
3,25307,11/1/2017,Trash - Curbside/Alley Missed Pickup,missed,2603 old matthews rd,37207.0,WASTE IND,2206,2.0,1735691.771,685027.2459
4,25312,11/1/2017,Trash - Curbside/Alley Missed Pickup,Missed the even side of the road.,604 croley dr,37209.0,RED RIVER,4203,20.0,1710185.772,664205.1011
...,...,...,...,...,...,...,...,...,...,...,...
20221,267125,11/1/2019,Trash - Curbside/Alley Missed Pickup,MISSED...NEIGHBORS MISSED,2731 Murfreesboro Pike,37013.0,RED RIVER,4502,32.0,1781137.263,632448.5511
20222,267126,11/1/2019,Trash - Curbside/Alley Missed Pickup,entire alley,"1621 Long Ave, Nashville, TN 37206, United States",37206.0,METRO,9508,6.0,1749711.399,669201.6016
20223,267130,11/1/2019,Trash - Curbside/Alley Missed Pickup,missed several,"2943 Windemere Cir, Nashville, TN 37214, Unite...",37214.0,RED RIVER,1502,15.0,1770293.388,674936.3038
20224,267134,11/1/2019,Trash - Curbside/Alley Missed Pickup,Caller stated trash was missed & were only pic...,"3325 Murfreesboro Pike, Nashville, TN 37013, U...",37013.0,RED RIVER,4502,32.0,1785224.998,627146.4002


In [4]:
trash_haulers = trash_haulers.rename(columns={'Request ': 'Request'})

In [5]:
trash_haulers['Zip Code'] = trash_haulers['Zip Code'].astype('Int64').astype(str)

In [6]:
trash_haulers['Trash Hauler'] = trash_haulers['Trash Hauler'].str.upper()

#### Standarizing street names

In [7]:
trash_haulers['incident_street'] = trash_haulers['Incident Address'].str.split(',', expand = True)[0].str.upper()

In [8]:
trash_haulers['incident_street'] = trash_haulers['incident_street'].str.replace('\bROAD$', 'RD', regex=True)

In [9]:
trash_haulers['incident_street'] = trash_haulers['incident_street'].str.replace('\bCOURT$', 'CT', regex=True)

In [10]:
trash_haulers['incident_street'] = trash_haulers['incident_street'].str.replace('\bDRIVE$', 'DR', regex=True)

#### Making Unique Street Identifier

In [11]:
trash_haulers['street_id'] = trash_haulers['incident_street'] + ' ' + trash_haulers['Zip Code']

### Subsetting missed pickups (`missed_pickups`)

In [12]:
missed_req = trash_haulers['Request'].value_counts().index[0]

In [13]:
missed_pickups = (trash_haulers.loc[
    (trash_haulers['Request'] == missed_req) | 
    (trash_haulers['Description'].str.contains('fail|miss|not pick|forgot', case=False, na=False))])

In [14]:
missed_pickups.groupby('Trash Hauler')['Request Number'].count()

Trash Hauler
METRO         3048
RED RIVER    12959
WASTE IND     1139
Name: Request Number, dtype: int64

### Subsetting Red River missed pickups

In [15]:
red_missed = missed_pickups.loc[missed_pickups['Trash Hauler'] == 'RED RIVER']

In [16]:
red_missed.street_id.value_counts().head(30)

street_id
12546 OLD HICKORY BLVD 37013      21
5135 HICKORY HOLLOW PKWY 37013    20
3710 N NATCHEZ CT 37211           19
6007 OBRIEN AVE 37209             19
1584 BELL RD 37211                18
320 OLD HICKORY BLVD 37221        17
802 CRESCENT RD 37205             17
607 ESTES RD 37215                16
1537 HARDING PL 37215             15
162 ANTIOCH PIKE 37211            14
3929 STEWARTS LN 37218            14
1601 S OBSERVATORY DR 37215       14
617 KINSEY BLVD 37115             13
14881 OLD HICKORY BLVD 37013      13
116 MARGARET ST 37115             13
2731 MURFREESBORO PIKE 37013      12
6301 HARDING PIKE 37205           12
209 PAGE RD 37205                 11
111 BARTON LN 37214               11
3116 ANDERSON RD 37013            10
120 BESS CT S 37013               10
200 BROOK HOLLOW RD 37205         10
115 BARTON LN 37214               10
4024 STEWARTS LN 37218            10
208 FLINT RIDGE CT 37189          10
14885 OLD HICKORY BLVD 37013      10
1916 S HAMILTON RD 37218    